In [4]:
import pandas as pd

games = pd.read_csv('../Data/processed/nba_games_2022_2026.csv')

In [5]:
games.head()

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE,SEASON
0,22022,1610612738,BOS,Boston Celtics,22200001,2022-10-18,BOS vs. PHI,W,240,46,...,36,24,8,3,11,24,126,9,1,2022-23
1,22022,1610612755,PHI,Philadelphia 76ers,22200001,2022-10-18,PHI @ BOS,L,240,40,...,31,16,8,3,14,25,117,-9,1,2022-23
2,22022,1610612744,GSW,Golden State Warriors,22200002,2022-10-18,GSW vs. LAL,W,240,45,...,48,31,11,4,18,23,123,14,1,2022-23
3,22022,1610612747,LAL,Los Angeles Lakers,22200002,2022-10-18,LAL @ GSW,L,240,40,...,48,23,12,4,22,18,109,-14,1,2022-23
4,22022,1610612753,ORL,Orlando Magic,22200003,2022-10-19,ORL @ DET,L,240,42,...,48,21,5,5,18,24,109,-4,1,2022-23


In [6]:
games.shape

(9840, 30)

In [7]:
rows_per_game = games.groupby("GAME_ID").size()

print(rows_per_game.value_counts())

2    4920
Name: count, dtype: int64


In [8]:
season_summary = games.groupby("SEASON").agg(
    rows=("GAME_ID", "size"),
    unique_games=("GAME_ID", "nunique"),
    teams=("TEAM_ID", "nunique")
)

season_summary["rows_per_game"] = (
    season_summary["rows"]
    / season_summary["unique_games"]
)

print(season_summary)

         rows  unique_games  teams  rows_per_game
SEASON                                           
2022-23  2460          1230     30            2.0
2023-24  2460          1230     30            2.0
2024-25  2460          1230     30            2.0
2025-26  2460          1230     30            2.0


In [9]:
games["GAME_DATE"] = pd.to_datetime(
    games["GAME_DATE"],
    errors="coerce"
)

print(games["GAME_DATE"].dtype)
print("Invalid dates:", games["GAME_DATE"].isna().sum())
print("Earliest:", games["GAME_DATE"].min())
print("Latest:", games["GAME_DATE"].max())

datetime64[ns]
Invalid dates: 0
Earliest: 2022-10-18 00:00:00
Latest: 2026-04-12 00:00:00


In [10]:
games = games.sort_values(
    ["GAME_DATE", "GAME_ID", "TEAM_ID"]
).reset_index(drop=True)

# vs. = home 
# @ = away

In [11]:
games["IS_HOME"] = (
    games["MATCHUP"]
    .str.contains("vs.", regex=False)
    .astype(int)
)

In [13]:
games["WIN"] = games["WL"].map({
    "W": 1,
    "L": 0
})

team_summary = games.groupby(
    ["SEASON", "TEAM_ABBREVIATION"]
).agg(
    games=("GAME_ID", "nunique"),
    wins=("WIN", "sum"),
    avg_points=("PTS", "mean"),
    avg_point_diff=("PLUS_MINUS", "mean")
)

team_summary["win_percentage"] = (
    team_summary["wins"]
    / team_summary["games"]
)

print(team_summary)

                           games  wins  avg_points  avg_point_diff  \
SEASON  TEAM_ABBREVIATION                                            
2022-23 ATL                   82    41  118.426829        0.292683   
        BKN                   82    45  113.353659        0.853659   
        BOS                   82    57  117.939024        6.524390   
        CHA                   82    27  110.951220       -6.243902   
        CHI                   82    40  113.121951        1.292683   
...                          ...   ...         ...             ...   
2025-26 SAC                   82    22  111.000000      -10.000000   
        SAS                   82    62  119.829268        8.304878   
        TOR                   82    46  114.634146        2.829268   
        UTA                   82    22  117.585366       -8.426829   
        WAS                   82    17  112.902439      -11.975610   

                           win_percentage  
SEASON  TEAM_ABBREVIATION                  
2

In [14]:
home_away_summary = games.groupby("IS_HOME").agg(
    games=("GAME_ID", "size"),
    win_rate=("WIN", "mean"),
    average_points=("PTS", "mean"),
    average_point_diff=("PLUS_MINUS", "mean")
)

print(home_away_summary)

         games  win_rate  average_points  average_point_diff
IS_HOME                                                     
0         4930  0.444625      113.577688           -2.008722
1         4910  0.555601      115.591446            2.016904


In [15]:
games["AVG_PTS_LAST_10"] = (
    games.groupby("TEAM_ID")["PTS"]
    .transform(
        lambda x: x.shift(1).rolling(10).mean()
    )
)

In [16]:
print(
    games.groupby(
        ["SEASON", "IS_HOME"]
    )["WIN"]
    .mean()
)

SEASON   IS_HOME
2022-23  0          0.419512
         1          0.580488
2023-24  0          0.456911
         1          0.543089
2024-25  0          0.455870
         1          0.544490
2025-26  0          0.446154
         1          0.554286
Name: WIN, dtype: float64


In [20]:
game_team_counts = games.groupby("GAME_ID").agg(
    row_count=("TEAM_ID", "size"),
    unique_teams=("TEAM_ID", "nunique")
)

print(game_team_counts.value_counts())

row_count  unique_teams
2          2               4920
Name: count, dtype: int64


In [21]:
problem_games = game_team_counts[
    (game_team_counts["row_count"] != 2)
    |
    (game_team_counts["unique_teams"] != 2)
]

print(problem_games)

Empty DataFrame
Columns: [row_count, unique_teams]
Index: []
